# Algoritmos de Regresión

In [5]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# 1. Iniciamos Spark en este nuevo notebook
spark = SparkSession.builder.appName("Semana12_Supervisados").getOrCreate()

# 2. Leemos DIRECTAMENTE los datos que guardamos en el Paso 1
ruta_datos = "/home/jovyan/work/autotec/trabajoneiel/Semana 10/modelos2/datos_etiquetados_kmeans"
df_clusters = spark.read.parquet(ruta_datos)

# 4. Mostramos la tabla para comprobar que TODO está ahí
df_clusters.show(10)

+-------+-----------+----+-----+--------------------+-----------+---------+----------+---------------+---------------+-----------------+--------------+--------------------+--------------------+----------+
| precio|kilometraje|anio|marca|              modelo|combustible|marca_idx|modelo_idx|combustible_idx|       marca_oh|        modelo_oh|combustible_oh|            features|      scaledFeatures|prediction|
+-------+-----------+----+-----+--------------------+-----------+---------+----------+---------------+---------------+-----------------+--------------+--------------------+--------------------+----------+
|2.299E7|    27294.0|2024| audi|A1 Sportback 30 T...|    bencina|     17.0|     161.0|            0.0|(76,[17],[1.0])|(804,[161],[1.0])| (4,[0],[1.0])|(887,[0,1,2,20,24...|(887,[0,1,2,20,24...|         0|
|2.299E7|    11766.0|2024| audi|A1 Sportback 30 T...|    bencina|     17.0|     161.0|            0.0|(76,[17],[1.0])|(804,[161],[1.0])| (4,[0],[1.0])|(887,[0,1,2,20,24...|(887,[0,

In [8]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
cols_a_borrar = ["marca_idx", "modelo_idx", "combustible_idx", "marca_oh", "modelo_oh", "combustible_oh"]
for c in cols_a_borrar:
    if c in df_clusters.columns:
        df_clusters = df_clusters.drop(c)
cat_cols = ["marca", "modelo", "combustible"]

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in cat_cols
]

encoder = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in cat_cols],
    outputCols=[f"{c}_oh" for c in cat_cols],
    handleInvalid="keep"
)

pipeline = Pipeline(stages=indexers + [encoder])
df_encoded = pipeline.fit(df_clusters).transform(df_clusters)

assembler_regresion = VectorAssembler(
    inputCols=["kilometraje", "anio"] + [f"{c}_oh" for c in cat_cols],
    outputCol="features_regresion"
)

df_vector_reg = assembler_regresion.transform(df_encoded)

scaler_reg = StandardScaler(
    inputCol="features_regresion",
    outputCol="scaledFeatures_regresion"
)
scaler_model_reg = scaler_reg.fit(df_vector_reg)
df_para_regresion = scaler_model_reg.transform(df_vector_reg)

df_para_regresion = df_para_regresion.withColumnRenamed("precio", "label_precio").drop("prediction")

train_reg, test_reg = df_para_regresion.randomSplit([0.7, 0.3], seed=42)

In [9]:
# Configurar el modelo de Regresión Lineal
lr_regresion = LinearRegression(
    featuresCol="scaledFeatures_regresion", 
    labelCol="label_precio", 
    maxIter=10
)

# Entrenar el modelo con los datos de entrenamiento
lr_reg_model = lr_regresion.fit(train_reg)

# Hacer las predicciones sobre los datos de prueba
predictions_regresion = lr_reg_model.transform(test_reg)

# Mostrar las predicciones junto al precio real
print("=== COMPARATIVA: PRECIO REAL VS PRECIO PREDICHO ===")
predictions_regresion.select("kilometraje", "label_precio", "prediction").show(10)

=== COMPARATIVA: PRECIO REAL VS PRECIO PREDICHO ===
+-----------+------------+--------------------+
|kilometraje|label_precio|          prediction|
+-----------+------------+--------------------+
|    45000.0|   2000000.0| -2435311.1432294846|
|   103000.0|   2790000.0|   6611178.520380974|
|   180000.0|   2950000.0| -1443612.7794322968|
|   221000.0|   3150000.0|  -6580496.085532665|
|   120000.0|   3550000.0|    547662.264901638|
|   132000.0|   3690000.0|   3026692.417784214|
|   235000.0|   3980000.0|  1298823.9734048843|
|   198000.0|   4500000.0|  -2458724.593861103|
|    89000.0|   4980000.0|1.2392798846565723E7|
|   195000.0|   5000000.0|  5439881.5576581955|
+-----------+------------+--------------------+
only showing top 10 rows



In [10]:
# Configurar los evaluadores de regresión
evaluator_r2 = RegressionEvaluator(labelCol="label_precio", predictionCol="prediction", metricName="r2")
evaluator_rmse = RegressionEvaluator(labelCol="label_precio", predictionCol="prediction", metricName="rmse")

r2 = evaluator_r2.evaluate(predictions_regresion)
rmse = evaluator_rmse.evaluate(predictions_regresion)

print("==================================================")
print("     EVALUACIÓN DE LA REGRESIÓN (SEMANA 12)       ")
print("==================================================")
print(f"R² (Coeficiente de Determinación): {r2 * 100:.2f}%")
print(f"RMSE (Error promedio del modelo):  {rmse:.4f}")
print("==================================================")

     EVALUACIÓN DE LA REGRESIÓN (SEMANA 12)       
R² (Coeficiente de Determinación): 52.13%
RMSE (Error promedio del modelo):  6888990.2260


In [11]:
# Imprimir la intersección (b) y los coeficientes (m)
print(f"Intersección (Precio base): {lr_reg_model.intercept:.4f}")
print(f"Coeficiente de 'kilometraje':    {lr_reg_model.coefficients[0]:.4f}")
print(f"Coeficiente de 'anio': {lr_reg_model.coefficients[1]:.4f}")

Intersección (Precio base): -2345075237.4076
Coeficiente de 'kilometraje':    -1207419.4266
Coeficiente de 'anio': 4155825.6360
